In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider, HBox, Layout, VBox, HTML
from IPython.display import display

display(HTML("<style>div.output_scroll { max-height: none !important; }</style>"))


def plot_filter_specs(epsilon=0.4, A=10.0, sr=0.3, wp=2.0, ws=3.5):
    fig, ax = plt.subplots(figsize=(10, 6))
    omega = np.linspace(0.01, 6.0, 1000)
    n = 4

    def T_n(n, x): return np.where(x <= 1.0, np.cos(n * np.arccos(np.clip(x, -1.0, 1.0))), np.cosh(n * np.arccosh(np.maximum(x, 1.0))))

    H_mag = np.zeros_like(omega)

    for i, w in enumerate(omega):
        if w <= wp:
            H_mag[i] = 1.0 / np.sqrt(1.0 + (epsilon * T_n(n, w / wp))**2)
        elif w < ws:
            val_wp = 1.0 / np.sqrt(1.0 + epsilon**2)
            val_ws = 1.0 / A
            ratio = (w - wp) / (ws - wp)
            H_mag[i] = val_wp - ratio * (val_wp - val_ws)
        else:
            val = (1.0 / A) * (1.0 + sr * np.abs(np.sin(4 * (w - ws)))) / (1.0 + sr)
            H_mag[i] = min(val, 1.0 / A)

    ax.plot(omega, H_mag, 'r-', linewidth=2, label=r'$|H(e^{j\omega})|$')

    upper_bound_pb = 1.0
    lower_bound_pb = 1.0 / np.sqrt(1.0 + epsilon**2)
    stop_bound = 1.0 / A

    ax.hlines(upper_bound_pb, 0, wp, colors='k', linewidth=1.5)
    ax.hlines(lower_bound_pb, 0, wp, colors='k', linewidth=1.5, linestyle='--')
    ax.hlines(stop_bound, ws, omega[-1], colors='k', linewidth=1.5)

    ax.axvline(wp, ymin=0, ymax=0.7, color='k', linestyle='--', linewidth=1)
    ax.axvline(ws, ymin=0, ymax=0.3, color='k', linestyle='--', linewidth=1)

    ax.fill_between([0, wp], upper_bound_pb, 1.2, color='blue', alpha=0.1, hatch='//')
    ax.fill_between([0, wp], lower_bound_pb, upper_bound_pb, color='blue', alpha=0.05)
    ax.fill_between([ws, omega[-1]], stop_bound, 1.0, color='blue', alpha=0.1, hatch='//')

    ax.set_xlim(0, max(omega))
    ax.set_ylim(-0.02, 1.25)
    ax.set_xlabel(r'$\omega$', fontsize=14)
    ax.set_ylabel(r'$|H(e^{j\omega})|$', fontsize=14)
    ax.set_xticks([0, wp, ws])
    ax.set_xticklabels(['0', r'$\omega_p$', r'$\omega_s$'], fontsize=12)

    y_ticks = [stop_bound, lower_bound_pb, 1.0]
    y_labels = [r'$\frac{1}{A}$', r'$\frac{1}{\sqrt{1+\varepsilon^2}}$', '1']
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels, fontsize=12)

    ax.text(wp / 2, 0.5, 'Passband', color='green', fontsize=12, fontweight='bold', ha='center')
    ax.text((wp + ws) / 2, 0.05, 'Transition\nband', color='black', fontsize=10, ha='center')
    ax.text((ws + omega[-1]) / 2, 0.5, 'Stopband', color='green', fontsize=12, fontweight='bold', ha='center')

    ax.grid(True, linestyle=':', alpha=0.6)
    plt.title('Normalized Frequency Response of a Low-Pass Analog Filter', fontsize=13, pad=15)
    plt.show()


slider_layout = Layout(width='260px')
style_opts = {'description_width': '55px'}

eps_slider = FloatSlider(min=0.1, max=1.5, step=0.05, value=0.4, description='ε:', style=style_opts, layout=slider_layout)
A_slider = FloatSlider(min=5.0, max=30.0, step=1.0, value=10.0, description='A:', style=style_opts, layout=slider_layout)
sr_slider = FloatSlider(min=0.0, max=0.8, step=0.05, value=0.3, description='δs:', style=style_opts, layout=slider_layout)
wp_slider = FloatSlider(min=1.0, max=2.5, step=0.1, value=2.0, description='ωp:', style=style_opts, layout=slider_layout)
ws_slider = FloatSlider(min=2.8, max=4.5, step=0.1, value=3.5, description='ωs:', style=style_opts, layout=slider_layout)

widget_plot = interactive(plot_filter_specs, epsilon=eps_slider, A=A_slider, sr=sr_slider, wp=wp_slider, ws=ws_slider)

theory_html = HTML("""
<div style="font-family: monospace; font-size: 13px; line-height: 1.5; margin-bottom: 8px;">
<b>ε:</b> Controls the ripple amplitude in the passband.<br>
<b>A:</b> Defines the minimum attenuation factor in the stopband.<br>
<b>δs:</b> Determines the ripple variation in the stopband.<br>
<b>ωp:</b> Represents the passband edge frequency.<br>
<b>ωs:</b> Represents the stopband edge frequency.
</div>
""")

controls = VBox([eps_slider, A_slider, sr_slider, wp_slider, ws_slider], layout=Layout(width='280px', justify_content='center'))

main_layout = HBox([widget_plot.children[-1], controls], layout=Layout(width='1100px', align_items='center', justify_content='center'))

display(VBox([theory_html, main_layout], layout=Layout(width='1100px')))